<a href="https://colab.research.google.com/github/Chathuranga630/ML-Workshop_SLT/blob/main/UAV_Fleet_Coordination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Basic UAV Fleet Coordination Simulation
---------------------------------------
Python 3.10+; dependencies: numpy, pandas, matplotlib

Models a fleet of UAVs assigning and completing independent tasks under a deadline.
It compares normal operation with false battery reports, spoofed messages, task
non-cooperation/failure, and a combined attack scenario.

This is a deliberately simple, explainable starting point—not a flight-physics model.
"""

from dataclasses import dataclass, asdict
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 17
N_UAVS = 8
N_TASKS = 12
MISSION_DEADLINE = 45.0  # simulated time units
INITIAL_BATTERY_RANGE = (75.0, 100.0)  # energy units


@dataclass
class UAV:
    uid: int
    battery: float
    true_battery: float
    speed: float
    energy_per_distance: float
    reliability: float = 1.0
    assigned_tasks: int = 0
    completed_tasks: int = 0
    failed_tasks: int = 0
    suspicious_events: int = 0
    active: bool = True


@dataclass
class Task:
    tid: int
    distance: float
    workload: float
    deadline: float
    status: str = "pending"
    assigned_to: Optional[int] = None
    start_time: Optional[float] = None
    finish_time: Optional[float] = None
    attempts: int = 0


def make_fleet(rng: np.random.Generator) -> List[UAV]:
    fleet = []
    for i in range(N_UAVS):
        battery = float(rng.uniform(*INITIAL_BATTERY_RANGE))
        fleet.append(UAV(
            uid=i, battery=battery, true_battery=battery,
            speed=float(rng.uniform(1.0, 1.5)),
            energy_per_distance=float(rng.uniform(0.75, 1.15)),
            reliability=float(rng.uniform(0.82, 0.99)),
        ))
    return fleet


def make_tasks(rng: np.random.Generator) -> List[Task]:
    return [
        Task(
            tid=i,
            distance=float(rng.uniform(5.0, 18.0)),
            workload=float(rng.uniform(3.0, 9.0)),
            deadline=float(rng.uniform(27.0, MISSION_DEADLINE)),
        )
        for i in range(N_TASKS)
    ]


def task_cost(uav: UAV, task: Task):
    # Workload adds processing/hover energy and time.
    duration = task.distance / uav.speed + 0.8 * task.workload
    energy = task.distance * uav.energy_per_distance + 0.35 * task.workload
    return duration, energy


def choose_uav(fleet: List[UAV], task: Task, now: float, unavailable: set,
              available_at: Dict[int, float]):
    """Greedy selection: minimize energy among deadline-feasible UAVs."""
    candidates = []
    for u in fleet:
        if not u.active or u.uid in unavailable:
            continue
        duration, energy = task_cost(u, task)
        start = max(now, available_at[u.uid])
        if start + duration > min(task.deadline, MISSION_DEADLINE):
            continue
        if u.battery < energy * 1.08:
            continue
        score = energy + 0.12 * duration
        candidates.append((score, duration, energy, start, u))
    if not candidates:
        return None
    _, duration, energy, start, selected = min(candidates, key=lambda x: x[0])
    return selected, duration, energy, start


def run_scenario(scenario: str, seed: int = SEED):
    rng = np.random.default_rng(seed)
    fleet = make_fleet(rng)
    tasks = make_tasks(rng)
    log = []
    total_energy = 0.0
    spoofed_rejected = 0
    reassigned = 0
    available_at = {u.uid: 0.0 for u in fleet}
    compromised = {1, 5}
    false_battery = scenario in {"false_battery", "combined"}
    spoofed_messages = scenario in {"spoofed_messages", "combined"}
    noncooperation = scenario in {"noncooperation", "combined"}

    # Spoofed completion/status messages are rejected unless authenticated.
    if spoofed_messages:
        for task in tasks:
            if task.tid % 3 == 0:
                spoofed_rejected += 1
                log.append({"event": "spoof_rejected", "task_id": task.tid,
                            "uav_id": -1, "time": 0.0, "energy": 0.0})

    # Process tighter-deadline tasks first; UAVs can work concurrently.
    tasks.sort(key=lambda t: t.deadline)
    for task in tasks:
        tried = set()
        success = False
        while True:
            # Simulate false battery claims on compromised UAVs before selection.
            if false_battery:
                for u in fleet:
                    if u.uid in compromised and u.uid not in tried:
                        u.battery = min(125.0, u.true_battery + 25.0)

            choice = choose_uav(fleet, task, 0.0, tried, available_at)
            if choice is None:
                break
            uav, duration, energy, start_time = choice
            tried.add(uav.uid)
            task.attempts += 1
            uav.assigned_tasks += 1

            # Cross-check reported battery against the coordinator's energy ledger.
            if abs(uav.battery - uav.true_battery) > 12.0:
                uav.suspicious_events += 1
                uav.battery = uav.true_battery
                log.append({"event": "false_battery_detected", "task_id": task.tid,
                            "uav_id": uav.uid, "time": start_time, "energy": 0.0})
                # Do not trust this report; re-evaluate using the corrected estimate.
                if uav.true_battery < energy * 1.08:
                    continue

            # A non-cooperating UAV accepts a task but sometimes fails to complete it.
            fails = noncooperation and uav.uid in compromised and rng.random() < 0.60
            if fails:
                wasted = min(uav.true_battery, 0.15 * energy)
                uav.true_battery -= wasted
                uav.battery = uav.true_battery
                total_energy += wasted
                uav.failed_tasks += 1
                uav.suspicious_events += 1
                log.append({"event": "task_failure", "task_id": task.tid,
                            "uav_id": uav.uid, "time": start_time, "energy": wasted})
                reassigned += 1
                continue

            finish_time = start_time + duration
            if finish_time > min(task.deadline, MISSION_DEADLINE):
                continue

            actual_energy = min(uav.true_battery, energy)
            uav.true_battery -= actual_energy
            uav.battery = uav.true_battery
            total_energy += actual_energy
            available_at[uav.uid] = finish_time
            uav.completed_tasks += 1
            task.status = "completed"
            task.assigned_to = uav.uid
            task.start_time = start_time
            task.finish_time = finish_time
            log.append({"event": "task_completed", "task_id": task.tid,
                        "uav_id": uav.uid, "time": finish_time, "energy": actual_energy})
            success = True
            break

        if not success:
            task.status = "missed"
            log.append({"event": "task_missed", "task_id": task.tid,
                        "uav_id": -1, "time": min(MISSION_DEADLINE, task.deadline),
                        "energy": 0.0})

    completed = sum(t.status == "completed" for t in tasks)
    mission_time = max((t.finish_time or 0.0 for t in tasks), default=0.0)
    result = {
        "scenario": scenario,
        "tasks_total": len(tasks),
        "tasks_completed": completed,
        "tasks_missed": len(tasks) - completed,
        "completion_rate_pct": 100.0 * completed / len(tasks),
        "energy_used": round(total_energy, 2),
        "energy_per_completed_task": round(total_energy / completed, 2) if completed else np.nan,
        "mission_time": round(mission_time, 2),
        "spoofed_messages_rejected": spoofed_rejected,
        "false_battery_reports_detected": sum(x["event"] == "false_battery_detected" for x in log),
        "task_failures_detected": sum(x["event"] == "task_failure" for x in log),
        "tasks_reassigned_after_failure": reassigned,
        "uavs_flagged": sum(u.suspicious_events > 0 for u in fleet),
    }
    return result, pd.DataFrame(log), pd.DataFrame([asdict(u) for u in fleet]), tasks

def main():
    scenarios = ["baseline", "false_battery", "spoofed_messages",
                 "noncooperation", "combined"]
    results, logs, uav_tables = [], [], []
    for scenario in scenarios:
        result, log, uavs, _ = run_scenario(scenario)
        results.append(result)
        log.insert(0, "scenario", scenario)
        logs.append(log)
        uavs.insert(0, "scenario", scenario)
        uav_tables.append(uavs)

    results_df = pd.DataFrame(results)
    logs_df = pd.concat(logs, ignore_index=True)
    uavs_df = pd.concat(uav_tables, ignore_index=True)

    results_df.to_csv("scenario_results.csv", index=False)
    logs_df.to_csv("event_log.csv", index=False)
    uavs_df.to_csv("uav_final_state.csv", index=False)

    print("\n=== Scenario comparison ===")
    print(results_df.to_string(index=False))
    print("\nSaved: scenario_results.csv, event_log.csv, uav_final_state.csv")

    # Figure 1: completion rate by scenario
    ax = results_df.plot(x="scenario", y="completion_rate_pct", kind="bar", legend=False)
    ax.set_ylabel("Tasks completed (%)")
    ax.set_xlabel("Scenario")
    ax.set_title("Mission completion rate by scenario")
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.savefig("completion_rate.png", dpi=160)
    plt.close()

    # Figure 2: total energy by scenario
    ax = results_df.plot(x="scenario", y="energy_used", kind="bar", legend=False)
    ax.set_ylabel("Energy units")
    ax.set_xlabel("Scenario")
    ax.set_title("Total energy used by scenario")
    plt.tight_layout()
    plt.savefig("energy_used.png", dpi=160)
    plt.close()

    # Figure 3: attack detection / recovery counts
    plot_cols = ["spoofed_messages_rejected", "false_battery_reports_detected",
                 "task_failures_detected", "tasks_reassigned_after_failure"]
    ax = results_df.set_index("scenario")[plot_cols].plot(kind="bar")
    ax.set_ylabel("Count")
    ax.set_title("Detection and recovery events")
    plt.tight_layout()
    plt.savefig("detection_recovery.png", dpi=160)
    plt.close()
    print("Saved: completion_rate.png, energy_used.png, detection_recovery.png")


if __name__ == "__main__":
    main()



=== Scenario comparison ===
        scenario  tasks_total  tasks_completed  tasks_missed  completion_rate_pct  energy_used  energy_per_completed_task  mission_time  spoofed_messages_rejected  false_battery_reports_detected  task_failures_detected  tasks_reassigned_after_failure  uavs_flagged
        baseline           12               12             0                100.0       127.15                       10.6         42.94                          0                               0                       0                               0             0
   false_battery           12               12             0                100.0       127.15                       10.6         42.94                          0                               0                       0                               0             0
spoofed_messages           12               12             0                100.0       127.15                       10.6         42.94                          4              